In [1]:
import pandas as pd

In [3]:
# ---------------------------------------------------------------------
# ধাপ ০: ফাইল লোড করুন (শুধু দরকারি কলাম + optimized dtype, মেমোরি বাঁচাতে)
# ---------------------------------------------------------------------
INPUT_FILE = "dataset_accepted_loan.csv"  # <-- আপনার ফাইলের নাম/path বসান

usecols = [
    "id",                    # লোন আইডি (প্রাইমারি কি হিসেবে ব্যবহার হবে)
    "member_id",             # চেক করার জন্য রাখা হলো — সাধারণত সম্পূর্ণ ফাঁকা থাকে
    "annual_inc",
    "emp_length",
    "home_ownership",
    "addr_state",
    "loan_amnt",
    "term",
    "int_rate",
    "grade",
    "sub_grade",
    "purpose",
    "issue_d",
    "installment",
    "fico_range_low",
    "fico_range_high",
    "delinq_2yrs",
    "open_acc",
    "pub_rec",
    "revol_util",
    "dti",
    "loan_status",
    "last_pymnt_d",
    "total_pymnt",
    # ---- নতুন যোগ হওয়া ৭টা কলাম (Query ১১-১৮ এর জন্য) ----
    "verification_status",   # -> borrowers   (Query ১১)
    "application_type",      # -> borrowers   (Query ১৫)
    "revol_bal",              # -> credit_history (Query ১৩)
    "total_acc",              # -> credit_history (Query ১৩)
    "mort_acc",                # -> credit_history (Query ১৪)
    "pub_rec_bankruptcies",   # -> credit_history (Query ১৪)
    "inq_last_6mths",          # -> credit_history (Query ১৬)
]

dtype_map = {
    # "id" ইচ্ছাকৃতভাবে এখানে নেই — CSV-এর শেষে টেক্সট footer row (যেমন
    # "Total amount funded in policy code 1: ...") থাকতে পারে, যেটা জোর করে
    # int64 বানাতে গেলে read_csv-ই ভেঙে পড়ে। তাই আগে string হিসেবে লোড করে
    # পরে numeric coerce করা হবে (নিচে দেখুন)।
    "loan_amnt": "float32",
    "int_rate": "float32",
    "installment": "float32",
    "annual_inc": "float32",
    "dti": "float32",
    "fico_range_low": "float32",
    "fico_range_high": "float32",
    "delinq_2yrs": "float32",
    "open_acc": "float32",
    "pub_rec": "float32",
    "total_pymnt": "float32",
    "revol_bal": "float32",
    "total_acc": "float32",
    "mort_acc": "float32",
    "pub_rec_bankruptcies": "float32",
    "inq_last_6mths": "float32",
}

df = pd.read_csv(INPUT_FILE, usecols=usecols, dtype=dtype_map, low_memory=False)

print(f"মোট রো লোড হয়েছে (footer/garbage row বাদ দেওয়ার আগে): {len(df):,}")

মোট রো লোড হয়েছে (footer/garbage row বাদ দেওয়ার আগে): 2,260,701


In [4]:
# ---------------------------------------------------------------------
# ধাপ ১: সাফসুতরো করা (basic cleaning)
# ---------------------------------------------------------------------
# id কলাম এখন string/object — সংখ্যায় কনভার্ট করার চেষ্টা করুন, যা সংখ্যা না
# (footer/summary টেক্সট) সেগুলো NaN হয়ে যাবে এবং dropna দিয়ে বাদ পড়বে
df["id"] = pd.to_numeric(df["id"], errors="coerce")

rows_before = len(df)
df = df.dropna(subset=["id", "loan_status"])
rows_dropped = rows_before - len(df)
print(f"অ-সংখ্যা/ফাঁকা id বা loan_status থাকায় বাদ পড়েছে: {rows_dropped:,} রো")

df["id"] = df["id"].astype("int64")

# member_id সাধারণত privacy কারণে ফাঁকা থাকে — নিশ্চিত করে নিন
member_id_filled = df["member_id"].notna().sum()
print(f"member_id-তে ভরা রো সংখ্যা: {member_id_filled:,} (সাধারণত এটা 0 হয়)")

অ-সংখ্যা/ফাঁকা id বা loan_status থাকায় বাদ পড়েছে: 33 রো
member_id-তে ভরা রো সংখ্যা: 0 (সাধারণত এটা 0 হয়)


In [5]:
# ---------------------------------------------------------------------
# ধাপ ২: surrogate borrower_id বানানো
# ---------------------------------------------------------------------
# এই ডেটাসেটে আসল "একজন মানুষ একাধিক লোন নিয়েছে কিনা" ট্র্যাক করার উপায় নেই
# (member_id ফাঁকা থাকে বলে)। তাই প্রতিটা লোনকে একটা করে borrower ধরে নিয়ে
# loan_id-কেই borrower_id হিসেবে ব্যবহার করছি (1 loan = 1 borrower surrogate key)।
df["borrower_id"] = df["id"]

In [6]:
# ---------------------------------------------------------------------
# ধাপ ৩: চারটা টেবিলে ভাগ করা
# ---------------------------------------------------------------------

# --- Table 1: borrowers ---
borrowers = df[[
    "borrower_id", "annual_inc", "emp_length", "home_ownership", "addr_state", "dti",
    "verification_status", "application_type",
]].rename(columns={
    "annual_inc": "annual_income",
    "addr_state": "state",
})

# --- Table 2: loans ---
loans = df[[
    "id", "borrower_id", "loan_amnt", "term", "int_rate",
    "installment", "grade", "sub_grade", "purpose", "issue_d"
]].rename(columns={"id": "loan_id"})

# --- Table 3: credit_history ---
credit_history = df[[
    "borrower_id", "fico_range_low", "fico_range_high",
    "delinq_2yrs", "open_acc", "pub_rec", "revol_util",
    "revol_bal", "total_acc", "mort_acc", "pub_rec_bankruptcies", "inq_last_6mths",
]]

# --- Table 4: payment_status ---
payment_status = df[[
    "id", "loan_status", "last_pymnt_d", "total_pymnt"
]].rename(columns={"id": "loan_id"})

In [7]:
# ---------------------------------------------------------------------
# ধাপ ৪: CSV হিসেবে সেভ করা (এগুলোই পরে MySQL/PostgreSQL-এ import করবেন)
# ---------------------------------------------------------------------
borrowers.to_csv("borrowers.csv", index=False)
loans.to_csv("loans.csv", index=False)
credit_history.to_csv("credit_history.csv", index=False)
payment_status.to_csv("payment_status.csv", index=False)

print("৪টা টেবিল তৈরি হয়ে গেছে:")
print(f"  borrowers.csv       -> {len(borrowers):,} রো, {borrowers.shape[1]} কলাম")
print(f"  loans.csv           -> {len(loans):,} রো, {loans.shape[1]} কলাম")
print(f"  credit_history.csv  -> {len(credit_history):,} রো, {credit_history.shape[1]} কলাম")
print(f"  payment_status.csv  -> {len(payment_status):,} রো, {payment_status.shape[1]} কলাম")

৪টা টেবিল তৈরি হয়ে গেছে:
  borrowers.csv       -> 2,260,668 রো, 8 কলাম
  loans.csv           -> 2,260,668 রো, 10 কলাম
  credit_history.csv  -> 2,260,668 রো, 12 কলাম
  payment_status.csv  -> 2,260,668 রো, 4 কলাম
